# Day 2 notebook companion

Run in order with synthetic data. Mermaid diagrams render on the website. Setup and shared helpers are embedded; no checkout is required. Learner exercises report NOT ATTEMPTED until implemented. Reference checks are separate. Optional controls also have direct function calls.


In [ ]:
import importlib.metadata
import subprocess
import sys
for package, version in {"cryptography": "50.0.1", "matplotlib": "3.10.6", "ipywidgets": "8.1.7"}.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])
print("Dependencies ready. Restart if an older library was already imported, then run all cells.")


## Shared teaching helpers

Inspect this implementation. TLS uses real SSL objects over memory buffers and temporary test key files; no system trust changes or network listeners. The teaching KDF is not a standardized protocol key schedule.


In [ ]:
"""Day 2 teaching helpers. Real TLS over MemoryBIO; no sockets or trust-store changes."""
from datetime import datetime, timedelta, timezone
from pathlib import Path
import ssl
import tempfile
import hashlib
from cryptography import x509
from cryptography.x509.oid import NameOID, ExtendedKeyUsageOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


def make_pki(expired=False):
    """Create an isolated root, intermediate, server, and client for this run."""
    now = datetime.now(timezone.utc)
    keys = {name: ec.generate_private_key(ec.SECP256R1())
            for name in ('root', 'intermediate', 'server', 'client')}
    names = {name: x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'Workshop ' + name)])
             for name in keys}
    certs = {}
    for name, issuer, ca, path_length, eku in [
        ('root', 'root', True, 1, None),
        ('intermediate', 'root', True, 0, None),
        ('server', 'intermediate', False, None, ExtendedKeyUsageOID.SERVER_AUTH),
        ('client', 'intermediate', False, None, ExtendedKeyUsageOID.CLIENT_AUTH),
    ]:
        end = now - timedelta(days=1) if expired and name == 'server' else now + timedelta(days=7)
        builder = (x509.CertificateBuilder().subject_name(names[name]).issuer_name(names[issuer])
                   .public_key(keys[name].public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(now - timedelta(days=2)).not_valid_after(end)
                   .add_extension(x509.BasicConstraints(ca=ca, path_length=path_length), critical=True)
                   .add_extension(x509.KeyUsage(digital_signature=True, content_commitment=False,
                       key_encipherment=False, data_encipherment=False, key_agreement=False,
                       key_cert_sign=ca, crl_sign=ca, encipher_only=False, decipher_only=False), critical=True)
                   .add_extension(x509.SubjectKeyIdentifier.from_public_key(keys[name].public_key()), False)
                   .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(keys[issuer].public_key()), False))
        if eku:
            builder = builder.add_extension(x509.ExtendedKeyUsage([eku]), False)
            builder = builder.add_extension(x509.SubjectAlternativeName([
                x509.DNSName('invoice.test' if name == 'server' else 'client.test')]), False)
        certs[name] = builder.sign(keys[issuer], hashes.SHA256())
    return keys, certs


def tls_trial(hostname='invoice.test', trust_root=True, expired=False,
              include_intermediate=True, mtls=False, send_client=True,
              client_wrong_eku=False):
    """Handshake and exchange application bytes. Failures raise ssl.SSLError.

    Private PEM files are disposable teaching keys in a temporary directory.
    Does not implement online revocation, networking, or authorization policy.
    """
    keys, certs = make_pki(expired)
    pem = lambda c: c.public_bytes(serialization.Encoding.PEM)
    with tempfile.TemporaryDirectory(prefix='workshop-pki-') as directory:
        base = Path(directory)
        for name in ('server', 'client'):
            selected = 'server' if name == 'client' and client_wrong_eku else name
            chain = pem(certs[selected])
            if include_intermediate or name == 'client':
                chain += pem(certs['intermediate'])
            (base / (name + '.pem')).write_bytes(chain)
            (base / (name + '.key')).write_bytes(keys[selected].private_bytes(
                serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                serialization.NoEncryption()))
        server_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
        client_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        for context in (server_context, client_context):
            context.minimum_version = context.maximum_version = ssl.TLSVersion.TLSv1_3
        server_context.load_cert_chain(str(base / 'server.pem'), str(base / 'server.key'))
        if trust_root:
            client_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if mtls:
            server_context.verify_mode = ssl.CERT_REQUIRED
            server_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if send_client:
            client_context.load_cert_chain(str(base / 'client.pem'), str(base / 'client.key'))
        ci, co, si, so = (ssl.MemoryBIO() for _ in range(4))
        client = client_context.wrap_bio(ci, co, server_hostname=hostname)
        server = server_context.wrap_bio(si, so, server_side=True)
        completed = [False, False]

        def transfer():
            for outgoing, incoming in ((co, si), (so, ci)):
                if outgoing.pending:
                    incoming.write(outgoing.read())

        for _ in range(100):
            for index, peer in enumerate((client, server)):
                if not completed[index]:
                    try:
                        peer.do_handshake()
                        completed[index] = True
                    except ssl.SSLWantReadError:
                        pass
            transfer()
            if all(completed):
                break
        else:
            raise RuntimeError('TLS handshake stalled')
        payload = b'synthetic confidential invoice'
        client.write(payload)
        transfer()
        assert server.read(4096) == payload
        return {'version': client.version(), 'cipher': client.cipher()[0],
                'client_authenticated': bool(server.getpeercert()),
                'application_bytes': len(payload)}


def expect_rejection(operation, exceptions):
    """Assert the negative case, without accepting a silent failure."""
    try:
        operation()
    except exceptions:
        return
    raise AssertionError('Expected rejection did not occur')


def derive_day2(secret, transcript, direction=b'alice-to-bob'):
    """Teaching KDF only, not a standardized TLS or hybrid key schedule."""
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                info=b'workshop-day2:v1|' + hashlib.sha256(transcript).digest()
                + b'|' + direction).derive(secret)


# Session 6: PKI and Certificates

**60 minutes taught · 90–120 minutes independently.** Instructor (see course website) · Notebook (see course website)

## Outcomes and preparation

You will trace a certificate path to a locally trusted root, distinguish signature verification from service identity validation, and explain why a valid client certificate does not grant unrestricted access. Review Session 5's distinction between trusted keys and arbitrary supplied keys.

Use the Day 2 setup (see course website). The notebook embeds the shared teaching helpers; local demonstrations also include them. The snippets below run in order after those helpers. All certificates and keys are generated for this run and are unrelated to real identities.



## Certificates bind claims to keys

An X.509 certificate contains a public key, identity claims, validity interval, issuer, serial number, extensions, and the issuer's signature. The CA signs the certificate body. That signature protects the body from undetected change; it does not independently prove that the CA was entitled to assert the identity.

A relying party starts with local trust anchors and policy. A root is trusted because it was provisioned as trusted, not because it is self-signed. An attacker can generate a perfectly valid self-signed certificate. An intermediate allows a root to delegate issuing authority without using the root key for routine issuance.

```mermaid
flowchart TD
    T["Locally configured trust anchor"] --> R["Root CA"]
    R --> I["Intermediate CA: constrained issuing authority"]
    I --> S["Server certificate: invoice.test"]
    I --> C["Client certificate: workload identity"]
    S --> V["Validate chain, time, purpose and expected name"]
```

Read from the trust anchor down: a server sends enough intermediate certificates to help path building. Sending a root does not make the peer trust that root. Real environments can offer multiple possible paths and policies; our helper creates one simple hierarchy.

| Field or extension | Question it helps answer | Common mistake |
| --- | --- | --- |
| SAN | Does this certificate cover the expected DNS/IP identity? | Treating a matching display name as sufficient |
| Basic Constraints | May this key act as a CA, and with what path limit? | Accepting an ordinary leaf as an issuer |
| Key Usage / Extended Key Usage | Is this key/certificate usable for the required operation and purpose? | Using a server-only certificate as a client identity |
| Validity interval | Is validation time inside the permitted period? | Disabling time checks to fix clock or renewal failures |
| Issuer and signature | Which issuing key signed this body? | Trusting the issuer's text name without validating the path |

## Inspect before trusting


In [ ]:
keys, certs = make_pki()
leaf = certs['server']
names = leaf.extensions.get_extension_for_class(x509.SubjectAlternativeName).value
assert names.get_values_for_type(x509.DNSName) == ['invoice.test']
assert not leaf.extensions.get_extension_for_class(x509.BasicConstraints).value.ca
assert certs['intermediate'].extensions.get_extension_for_class(x509.BasicConstraints).value.path_length == 0
print('PASS: inspected SAN, leaf constraints and intermediate path limit')


Inspection is not validation. Parsing an attacker-controlled certificate succeeds for many untrusted certificates. Do not implement a validator by merely comparing issuer strings or checking one signature. Use a maintained path validator with a defined trust store and application policy.

## Validation is a sequence of decisions

```mermaid
flowchart TD
    C["Received certificate chain"] --> P["Build acceptable path to local anchor"]
    P --> T["Check signatures, constraints, time and purpose"]
    T --> N["Match independently expected service identity"]
    N --> R["Apply revocation and local policy"]
    R --> A["Continue authenticated protocol"]
    T --> F["Any failed required check: reject"]
    N --> F
```

The expected hostname comes from the service the application intended to contact, not from whatever name the certificate happens to contain. DNS SANs and IP SANs are different identity types. Our `.test` name is an isolated teaching identity, not a public service.

The helper uses Python's TLS verifier for a real handshake over memory buffers. Predict which cases fail without weakening verification:


In [ ]:
assert tls_trial()['version'] == 'TLSv1.3'
for settings in [dict(hostname='other.test'), dict(trust_root=False),
                 dict(expired=True), dict(include_intermediate=False)]:
    expect_rejection(lambda settings=settings: tls_trial(**settings), ssl.SSLError)
print('PASS: trusted chain succeeds; wrong name, missing trust, expiry and missing intermediate fail')


Missing intermediates fail in this isolated setup because no cache or issuer-fetching service supplies them. In another client, cached intermediates can hide a deployment problem. Test clean clients as well as existing installations.

## Client certificates and revocation

Mutual TLS authenticates both ends according to certificate policy. Server verification of a client's certificate does not usually involve checking that client's DNS hostname. The server instead maps the authenticated identity to application roles and permissions. A valid certificate for service A must not authorize all service B operations.


In [ ]:
assert tls_trial(mtls=True)['client_authenticated']
expect_rejection(lambda: tls_trial(mtls=True, send_client=False), ssl.SSLError)
expect_rejection(lambda: tls_trial(mtls=True, client_wrong_eku=True), ssl.SSLError)
print('PASS: mTLS requires an acceptable client certificate')


CRLs publish revocation information; OCSP supplies certificate-status responses. Deployment must define status freshness, availability, and behavior when information is unavailable. Short validity reduces exposure duration but does not make compromise disappear immediately. Our helper does **not** perform online revocation checking: a successful lab handshake must not be described as a full revocation assessment.

For the state actor, issuance systems, trust-store administration, renewal credentials and root/intermediate private keys are valuable targets. Protect who may issue which identities, constrain delegation, monitor issuance, and rehearse replacement. Installing another root expands trust and is a security decision, not a debugging fix.

## Practice and answers

1. Why does sending the root in the server chain not fix an unknown-root error?
2. A certificate is in date and signed by a trusted CA but names another service. Accept it?
3. Why is a server-auth certificate insufficient for our client-auth requirement?
4. What additional evidence is needed to claim a certificate is not revoked?

<details><summary>Worked answers</summary>
<ol><li>The peer must already trust an appropriate anchor through local policy; a received root cannot grant itself trust.</li><li>No. Path validity and intended service identity are separate checks.</li><li>Purpose constraints matter even when a signature and chain are valid.</li><li>A defined revocation policy and sufficiently fresh, trustworthy status evidence; this demo has neither online OCSP nor CRL evaluation.</li></ol>
</details>

Ready to continue: explain an unknown issuer, wrong SAN, expired certificate and unauthorized client as different failures. Next: TLS (see course website) and Lab 4 (see course website).

## Sources and scope

Reviewed 22 September 2026: [RFC 5280](https://www.rfc-editor.org/rfc/rfc5280), [Python ssl](https://docs.python.org/3/library/ssl.html). The notebook creates temporary private PEM files for the TLS API and removes the directory afterward; filesystem removal is not a secure-erasure guarantee.


In [ ]:
print("PASS: completed session-06-pki-certificates demonstrations; learner status is reported separately")
